# Housing Price Prediction

## Portfolio Project Summary

**Domain:** Real Estate Analytics  
**Project Type:** Regression  
**Repository Link:** https://github.com/sundaralingam48/house-price-prediction-ml

### Business Problem
Home buyers, sellers, and analysts need practical ways to estimate property value. This project predicts sale price using property characteristics.

### Project Summary
This project predicts housing prices using square footage, rooms, bathrooms, age, property quality, and neighborhood score.

### Skills Demonstrated
- Data cleaning and preparation
- Exploratory data analysis
- Feature engineering
- Model training and evaluation
- Business interpretation and recommendations
- Ethical consideration of data science results

> This notebook is self-contained and uses simulated data so it can run successfully in an employer-facing portfolio without requiring private or restricted data. The same workflow can be adapted to a real dataset.


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 50)


## 2. Create or Load Dataset

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(106)
n = 700

df = pd.DataFrame({
    'sqft': np.random.normal(50, 15, n),
    'bedrooms': np.random.normal(50, 15, n),
    'bathrooms': np.random.normal(50, 15, n),
    'age': np.random.normal(50, 15, n),
    'quality_score': np.random.normal(50, 15, n),
    'neighborhood_score': np.random.normal(50, 15, n)
})

for col in df.columns:
    df[col] = np.maximum(df[col], 1)

weights = np.array([1200, 1800, 1500, -700, 2200, 1600])
noise = np.random.normal(0, 10000, n)
df['sale_price'] = df[['sqft', 'bedrooms', 'bathrooms', 'age', 'quality_score', 'neighborhood_score']].values.dot(weights) + noise

df.head()

## 3. Exploratory Data Analysis

In [ ]:
print("Dataset shape:", df.shape)
display(df.describe().T)

target_col = 'sale_price'
print("\nTarget distribution / summary:")
display(df[target_col].describe())


In [ ]:
# Visualization 1: target distribution
target_col = 'sale_price'

plt.figure(figsize=(8, 5))
if df[target_col].nunique() <= 10:
    df[target_col].value_counts().sort_index().plot(kind="bar")
    plt.ylabel("Count")
else:
    df[target_col].plot(kind="hist", bins=30)
    plt.ylabel("Frequency")
plt.title(f"Distribution of {target_col}")
plt.xlabel(target_col)
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2: correlation with target
target_col = 'sale_price'
corr = df.corr(numeric_only=True)[target_col].drop(target_col).sort_values()

plt.figure(figsize=(8, 5))
corr.plot(kind="barh")
plt.title(f"Feature Correlation with {target_col}")
plt.xlabel("Correlation")
plt.tight_layout()
plt.show()


## 4. Model Training and Evaluation

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

features = ['sqft', 'bedrooms', 'bathrooms', 'age', 'quality_score', 'neighborhood_score']
target = 'sale_price'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": rmse,
        "R2": r2_score(y_test, preds)
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
print("Best model:", best_model_name)


## 5. Model Visualization

In [ ]:
preds = best_model.predict(X_test)

plt.figure(figsize=(7, 5))
plt.scatter(y_test, preds, alpha=0.7)
plt.title("Actual vs Predicted Values")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.tight_layout()
plt.show()


## 6. Feature Importance

In [ ]:
# Feature importance using permutation importance
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.title("Permutation Feature Importance")
plt.xlabel("Mean Importance")
plt.tight_layout()
plt.show()

display(importance_df.sort_values("importance", ascending=False))


## Business Interpretation and Recommendations

### Key Findings
The notebook compares multiple models and selects the best-performing model based on the most appropriate evaluation metric. The feature importance section identifies which variables most strongly influence the prediction.

### Recommendations
1. Use the model as a decision-support tool, not as the only decision-maker.
2. Monitor model performance over time as new data becomes available.
3. Review high-impact features for fairness, quality, and possible bias.
4. Build a simple dashboard or reporting layer so stakeholders can understand results.

### Ethical Considerations
This project should avoid using sensitive personal information unless it is necessary, allowed, and properly protected. Model results should be explainable to business users, and decisions should be reviewed for fairness and unintended impact.

### Future Improvements
- Replace simulated data with a real validated dataset.
- Add cross-validation and hyperparameter tuning.
- Store model outputs in a reusable report.
- Deploy the workflow as a dashboard, API, or scheduled analytics process.
